# CNN Smoke Test: One Training Round

This notebook follows `agent_instruction_under_5min_training.md` with local files in current dir.

Signal source: `cropped_traces_3000_7000.h5` (dataset key `traces`)
Noise source: `noise_traces_4358x4000.h5` (dataset key `traces`)

Both are already length-4000 windows from the current directory.

In [6]:
import json
import time
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

SEED = 123
torch.manual_seed(SEED)

DATA_DIR = Path.cwd()
ARTIFACT_DIR = DATA_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "signal_file": "signal_vacuum_sum_crop_4000x8000.h5",
    "noise_file": "noise_traces_4000x8000.h5",
    "n_signal": 4000,
    "n_noise": 4000,
    "trace_len": 8000,
    "train_frac": 0.70,
    "val_frac": 0.15,
    "test_frac": 0.15,
    "batch_size": 32,
    "epochs": 5,
    "n_runs": 3,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "eps": 1e-6,
    "layers": [1, 1, 1],
    "kernel_size": 7,
}

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("device:", DEVICE)
print(json.dumps(CFG, indent=2))


device: cuda
{
  "signal_file": "signal_vacuum_sum_crop_4000x8000.h5",
  "noise_file": "noise_traces_4000x8000.h5",
  "n_signal": 4000,
  "n_noise": 4000,
  "trace_len": 8000,
  "train_frac": 0.7,
  "val_frac": 0.15,
  "test_frac": 0.15,
  "batch_size": 32,
  "epochs": 5,
  "n_runs": 3,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "eps": 1e-06,
  "layers": [
    1,
    1,
    1
  ],
  "kernel_size": 7
}


In [7]:
# Step 1: Read signal/noise H5 sources and align training sample length
signal_path = DATA_DIR / CFG["signal_file"]
noise_path = DATA_DIR / CFG["noise_file"]

if not signal_path.exists():
    raise FileNotFoundError(f"Missing signal file: {signal_path}")
if not noise_path.exists():
    raise FileNotFoundError(f"Missing noise file: {noise_path}")

def read_h5_traces(path):
    with h5py.File(path, "r") as f:
        key = "traces" if "traces" in f else list(f.keys())[0]
        arr = np.asarray(f[key][:], dtype=np.float32)
    return key, arr

signal_key, signal_all = read_h5_traces(signal_path)
noise_key, noise_all = read_h5_traces(noise_path)

print("Signal file:", signal_path)
print("Signal key:", signal_key, "shape:", signal_all.shape, "dtype:", signal_all.dtype)
print("Noise file:", noise_path)
print("Noise key:", noise_key, "shape:", noise_all.shape, "dtype:", noise_all.dtype)

target_len = int(CFG["trace_len"])
if signal_all.shape[1] < target_len or noise_all.shape[1] < target_len:
    raise ValueError(
        f"Requested trace_len={target_len}, but got signal_len={signal_all.shape[1]}, noise_len={noise_all.shape[1]}"
    )

signal_all = signal_all[:, :target_len]
noise_all = noise_all[:, :target_len]

assert signal_all.shape[0] >= CFG["n_signal"]
assert noise_all.shape[0] >= CFG["n_noise"]
print("Aligned shapes -> signal:", signal_all.shape, "noise:", noise_all.shape)


Signal file: /home/dwong/DELight_mtr/PCA_dev/wk8/PC_interpretation/CNN_demo/signal_vacuum_sum_crop_4000x8000.h5
Signal key: traces shape: (4000, 8000) dtype: float32
Noise file: /home/dwong/DELight_mtr/PCA_dev/wk8/PC_interpretation/CNN_demo/noise_traces_4000x8000.h5
Noise key: traces shape: (4000, 8000) dtype: float32
Aligned shapes -> signal: (4000, 8000) noise: (4000, 8000)


### Data-loading note
- Traces are loaded from current directory H5 files using dataset key `traces`.
- Training arrays are `signal_all` and `noise_all`.
- Final tensor shape before model input channel expansion: `(N, 4000)`.
- Preprocessing used: per-trace z-score only (mean/std).

In [8]:
# Build balanced dataset helpers and reproducible splits
def split_indices(rng, n, train_frac=0.70, val_frac=0.15):
    n_train = int(round(n * train_frac))
    n_val = int(round(n * val_frac))
    n_test = n - n_train - n_val
    order = rng.permutation(n)
    return order[:n_train], order[n_train:n_train+n_val], order[n_train+n_val:n_train+n_val+n_test]

def zscore_per_trace(x, eps=1e-6):
    m = x.mean(axis=1, keepdims=True)
    s = x.std(axis=1, keepdims=True)
    return (x - m) / (s + eps)

def make_split(signal_all, noise_all, signal_idx, noise_idx, sig_ids, noi_ids, rng, eps):
    xs = signal_all[signal_idx[sig_ids]]
    xn = noise_all[noise_idx[noi_ids]]
    ys = np.ones((xs.shape[0],), dtype=np.float32)
    yn = np.zeros((xn.shape[0],), dtype=np.float32)
    x = np.concatenate([xs, xn], axis=0).astype(np.float32, copy=False)
    y = np.concatenate([ys, yn], axis=0)
    p = rng.permutation(x.shape[0])
    x = x[p]
    y = y[p]
    x = zscore_per_trace(x, eps=eps)
    x = x[:, None, :]
    return x, y

def prepare_run_datasets(run_seed):
    rng = np.random.default_rng(run_seed)

    signal_idx = rng.permutation(signal_all.shape[0])[:CFG["n_signal"]]
    noise_idx = rng.permutation(noise_all.shape[0])[:CFG["n_noise"]]

    sig_train, sig_val, sig_test = split_indices(rng, len(signal_idx), CFG["train_frac"], CFG["val_frac"])
    noi_train, noi_val, noi_test = split_indices(rng, len(noise_idx), CFG["train_frac"], CFG["val_frac"])

    prep_t0 = time.perf_counter()
    X_train, y_train = make_split(signal_all, noise_all, signal_idx, noise_idx, sig_train, noi_train, rng, CFG["eps"])
    X_val, y_val = make_split(signal_all, noise_all, signal_idx, noise_idx, sig_val, noi_val, rng, CFG["eps"])
    X_test, y_test = make_split(signal_all, noise_all, signal_idx, noise_idx, sig_test, noi_test, rng, CFG["eps"])
    dataset_prep_seconds = time.perf_counter() - prep_t0

    split_meta = {
        "seed": run_seed,
        "signal_file": str(signal_path),
        "noise_file": str(noise_path),
        "signal_indices": signal_idx.tolist(),
        "noise_indices": noise_idx.tolist(),
        "signal_split": {"train": sig_train.tolist(), "val": sig_val.tolist(), "test": sig_test.tolist()},
        "noise_split": {"train": noi_train.tolist(), "val": noi_val.tolist(), "test": noi_test.tolist()},
    }

    return (X_train, y_train, X_val, y_val, X_test, y_test, dataset_prep_seconds, split_meta)


In [9]:
# Import ResNet1D and define model builder for repeated runs
from pathlib import Path as _Path
import types

try:
    from resnet_1d import ResNet1D
except Exception:
    code = _Path("resnet_1d.py").read_text()
    replacement = "from typing import Any\nNormLayer = Any\n\ndef GroupNorm1DGetter(num_groups: int = 8):\n    def _mk(channels: int):\n        g = min(num_groups, channels)\n        while g > 1 and channels % g != 0:\n            g -= 1\n        return nn.GroupNorm(g, channels)\n    return _mk\n"
    code = code.replace("from ..norm import GroupNorm1DGetter, NormLayer", replacement)
    mod = types.ModuleType("resnet_1d_local")
    mod.__dict__["nn"] = nn
    exec(code, mod.__dict__)
    ResNet1D = mod.ResNet1D

def build_model():
    model = ResNet1D(
        in_channels=1,
        layers=CFG["layers"],
        classes=1,
        kernel_size=CFG["kernel_size"],
    )
    return model.to(DEVICE)

tmp_model = build_model()
param_count = int(sum(p.numel() for p in tmp_model.parameters()))
del tmp_model
print("parameter_count:", param_count)


parameter_count: 961857


In [10]:
# Multi-run training loop (loop-ready): run n_runs with different seeds
def eval_loader(model, loader, criterion):
    model.eval()
    losses = []
    ys = []
    probs = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True).view(-1, 1)
            logits = model(xb)
            loss = criterion(logits, yb)
            losses.append(float(loss.item()))
            ys.append(yb.detach().cpu().numpy().ravel())
            probs.append(torch.sigmoid(logits).detach().cpu().numpy().ravel())
    y_true = np.concatenate(ys)
    y_prob = np.concatenate(probs)
    return float(np.mean(losses)), y_true, y_prob

all_run_summaries = []

for run_idx in range(CFG["n_runs"]):
    run_seed = SEED + run_idx
    torch.manual_seed(run_seed)

    (X_train, y_train, X_val, y_val, X_test, y_test, dataset_prep_seconds, split_meta) = prepare_run_datasets(run_seed)

    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
    test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=0)

    model = build_model()
    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["learning_rate"], weight_decay=CFG["weight_decay"])

    history = []
    total_t0 = time.perf_counter()
    train_time_sum = 0.0
    val_time_sum = 0.0

    print(f"\nRun {run_idx + 1}/{CFG['n_runs']} (seed={run_seed})")
    print(f"train/val/test: {X_train.shape[0]}/{X_val.shape[0]}/{X_test.shape[0]}")

    for epoch in range(1, CFG["epochs"] + 1):
        model.train()
        t0 = time.perf_counter()
        train_losses = []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True).view(-1, 1)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.item()))
        t1 = time.perf_counter()
        train_time_sum += (t1 - t0)

        v0 = time.perf_counter()
        val_loss, y_val_true, y_val_prob = eval_loader(model, val_loader, criterion)
        v1 = time.perf_counter()
        val_time_sum += (v1 - v0)

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "val_loss": val_loss,
            "val_roc_auc": float(roc_auc_score(y_val_true, y_val_prob)),
            "val_pr_auc": float(average_precision_score(y_val_true, y_val_prob)),
        }
        history.append(row)
        print(row)

    test_loss, y_test_true, y_test_prob = eval_loader(model, test_loader, criterion)
    y_test_pred = (y_test_prob >= 0.5).astype(np.int64)
    test_cm = confusion_matrix(y_test_true.astype(np.int64), y_test_pred).tolist()

    total_wall_seconds = time.perf_counter() - total_t0

    metrics = {
        "run_idx": run_idx,
        "run_seed": run_seed,
        "test_loss": float(test_loss),
        "test_roc_auc": float(roc_auc_score(y_test_true, y_test_prob)),
        "test_pr_auc": float(average_precision_score(y_test_true, y_test_prob)),
        "test_confusion_matrix_threshold_0_5": test_cm,
        "parameter_count": int(sum(p.numel() for p in model.parameters())),
        "device": str(DEVICE),
        "train_size": int(X_train.shape[0]),
        "val_size": int(X_val.shape[0]),
        "test_size": int(X_test.shape[0]),
    }

    timing = {
        "dataset_prep_seconds": float(dataset_prep_seconds),
        "training_seconds": float(train_time_sum),
        "validation_seconds": float(val_time_sum),
        "total_wall_seconds": float(total_wall_seconds),
    }

    run_tag = f"run_{run_idx:02d}"
    (ARTIFACT_DIR / f"splits_{run_tag}.json").write_text(json.dumps(split_meta, indent=2))
    (ARTIFACT_DIR / f"history_{run_tag}.json").write_text(json.dumps(history, indent=2))
    (ARTIFACT_DIR / f"metrics_{run_tag}.json").write_text(json.dumps(metrics, indent=2))
    (ARTIFACT_DIR / f"timing_{run_tag}.json").write_text(json.dumps(timing, indent=2))

    all_run_summaries.append({
        "run_idx": run_idx,
        "run_seed": run_seed,
        "val_last_roc_auc": float(history[-1]["val_roc_auc"]),
        "val_last_pr_auc": float(history[-1]["val_pr_auc"]),
        "test_roc_auc": float(metrics["test_roc_auc"]),
        "test_pr_auc": float(metrics["test_pr_auc"]),
    })

summary_path = ARTIFACT_DIR / "summary_runs.json"
summary_path.write_text(json.dumps(all_run_summaries, indent=2))

print("\nSaved multi-run summary:", summary_path)
for row in all_run_summaries:
    print(row)



Run 1/3 (seed=123)
train/val/test: 5600/1200/1200
{'epoch': 1, 'train_loss': 0.6826577956335885, 'val_loss': 0.6105053111126548, 'val_roc_auc': 0.8677194444444444, 'val_pr_auc': 0.8899221464022409}
{'epoch': 2, 'train_loss': 0.5360619587557657, 'val_loss': 0.4790609353467038, 'val_roc_auc': 0.8772027777777778, 'val_pr_auc': 0.900581247510577}
{'epoch': 3, 'train_loss': 0.47502446838787626, 'val_loss': 0.5328181284038644, 'val_roc_auc': 0.8827805555555557, 'val_pr_auc': 0.9054485446823023}
{'epoch': 4, 'train_loss': 0.4566565760544368, 'val_loss': 0.42617356698764, 'val_roc_auc': 0.8864305555555556, 'val_pr_auc': 0.9087534723692285}
{'epoch': 5, 'train_loss': 0.44858642901693074, 'val_loss': 0.41337427810618754, 'val_roc_auc': 0.887186111111111, 'val_pr_auc': 0.9094417975490312}

Run 2/3 (seed=124)
train/val/test: 5600/1200/1200
{'epoch': 1, 'train_loss': 0.6986247416904995, 'val_loss': 0.5654566774242803, 'val_roc_auc': 0.8558749999999999, 'val_pr_auc': 0.8840886178712284}
{'epoch': 2